# Text Generation Strategies

Comparing greedy, temperature, top-k, and top-p sampling using pretrained GPT-2.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import tiktoken
from minigpt.pretrained import load_gpt2
from minigpt.generate import generate

In [ ]:
model = load_gpt2('gpt2')
model.eval()
tokenizer = tiktoken.get_encoding('gpt2')

def gen(prompt, max_new_tokens=100, **kwargs):
    ids = torch.tensor([tokenizer.encode(prompt)])
    with torch.no_grad():
        out = generate(model, ids, max_new_tokens=max_new_tokens, context_length=1024, **kwargs)
    return tokenizer.decode(out[0].tolist())

print('Model loaded.')

## 1. Greedy vs Temperature

In [ ]:
prompt = 'The meaning of life is'

print('=== Greedy (temperature=0) ===')
print(gen(prompt))

print('\n=== Temperature=0.3 (conservative) ===')
print(gen(prompt, temperature=0.3))

print('\n=== Temperature=0.7 (balanced) ===')
print(gen(prompt, temperature=0.7))

print('\n=== Temperature=1.0 (full distribution) ===')
print(gen(prompt, temperature=1.0))

print('\n=== Temperature=2.0 (very random) ===')
print(gen(prompt, temperature=2.0))

## 2. Top-k Sampling

In [ ]:
prompt = 'Once upon a time'

print('=== Top-k=1 (greedy) ===')
print(gen(prompt, temperature=1.0, top_k=1))

print('\n=== Top-k=5 ===')
print(gen(prompt, temperature=1.0, top_k=5))

print('\n=== Top-k=40 (GPT-2 default) ===')
print(gen(prompt, temperature=1.0, top_k=40))

print('\n=== Top-k=200 ===')
print(gen(prompt, temperature=1.0, top_k=200))

## 3. Top-p (Nucleus) Sampling

In [ ]:
prompt = 'In a shocking discovery, scientists found that'

print('=== Top-p=0.1 (very focused) ===')
print(gen(prompt, temperature=1.0, top_p=0.1))

print('\n=== Top-p=0.5 ===')
print(gen(prompt, temperature=1.0, top_p=0.5))

print('\n=== Top-p=0.9 (common default) ===')
print(gen(prompt, temperature=1.0, top_p=0.9))

print('\n=== Top-p=0.99 ===')
print(gen(prompt, temperature=1.0, top_p=0.99))

## 4. Combined — The Sweet Spot

In practice, models use temperature + top-k + top-p together.

In [ ]:
prompt = 'Dear friend, I am writing to you because'

print('=== Greedy ===')
print(gen(prompt))

print('\n=== temp=0.7, top_k=40, top_p=0.9 (typical ChatGPT-style) ===')
print(gen(prompt, temperature=0.7, top_k=40, top_p=0.9))

print('\n=== temp=1.0, top_k=50, top_p=0.95 (more creative) ===')
print(gen(prompt, temperature=1.0, top_k=50, top_p=0.95))

## 5. Same Prompt, Multiple Runs

With sampling, every run gives a different output. Greedy always gives the same.

In [ ]:
prompt = 'The president announced that'

print('=== 3 greedy runs (should be identical) ===')
for i in range(3):
    print(f'Run {i+1}: {gen(prompt, max_new_tokens=30)}')

print('\n=== 3 sampled runs (should differ) ===')
for i in range(3):
    print(f'Run {i+1}: {gen(prompt, max_new_tokens=30, temperature=0.8, top_k=40)}')